# KMBank reusable template — k-means on any numeric bank matrix

**Short name:** `KMBank`

Swap the loader. Keep: scale mixed units, pick k from the business glossary, fit, profile centroids in original units, map names if a label exists, simulate knobs.

Works for: retail personas, branch mix, card-util bins, region NPL snapshots.


## Checklist

1. Rows = accounts / customers / branches. Drop ids and labels from `X`.
2. `StandardScaler` or a manual z-score. Fit the scaler on the matrix you cluster.
3. `k` from the story first, elbow second.
4. `KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_scaled)`.
5. `C_raw = centers * sd + mu` before you show a slide.
6. New row: same columns, same scaler, then `.predict`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from KMBank import KMeans, map_clusters_to_truth, adjusted_rand

df = pd.read_csv("data/bank_customers.csv")
label_col = "segment"                       # set None if unlabeled
id_col = "customer_id"
drop = [c for c in (label_col, id_col, "segment_name") if c and c in df.columns]
feature_cols = [c for c in df.columns if c not in drop]
X_raw = df[feature_cols].to_numpy(float)
y = None if label_col is None else df[label_col].to_numpy()

mu, sd = X_raw.mean(0), X_raw.std(0, ddof=0)
sd = np.where(sd == 0, 1.0, sd)
X = (X_raw - mu) / sd

k = 5
model = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
print("inertia", round(model.inertia_, 2), "sizes", np.bincount(model.labels_))
C_raw = model.cluster_centers_ * sd + mu
print(pd.DataFrame(C_raw, columns=feature_cols).round(2))

if y is not None:
    _, mapping, purity = map_clusters_to_truth(model.labels_, y, k)
    print("purity", round(purity, 4), "ARI", round(adjusted_rand(y, model.labels_), 4))
    print("map", mapping)


## Simulation stub

Change `k`, `n`, `noise` and record inertia / purity. See §13 of the solution notebook.